In [8]:
import time
import pandas as pd
from sklearn.metrics import accuracy_score
import xgboost as xgb
import lightgbm as lgb
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [2]:
results = []

In [3]:
import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/MedAssist AI/clean_190k_dataset.csv')

In [4]:
from sklearn.preprocessing import LabelEncoder
df_clean = df.copy()
# STEP 1: LABEL ENCODING
print("Translating diseases to numbers...")
encoder = LabelEncoder()
df_clean['target'] = encoder.fit_transform(df_clean['diseases'])
df_final = df_clean.drop(columns=['diseases'])

Translating diseases to numbers...


In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split

print("Isolating rare diseases to protect them from the split...")

# 1. Figure out which diseases are rare (1 row) and which are common (2+ rows)
class_counts = df_final['target'].value_counts()
rare_classes = class_counts[class_counts == 1].index
common_classes = class_counts[class_counts > 1].index

# 2. Split the dataset into two separate dataframes
df_rare = df_final[df_final['target'].isin(rare_classes)]
df_common = df_final[df_final['target'].isin(common_classes)]

# 3. Perform the Stratified Split ONLY on the common diseases
X_common = df_common.drop(columns=['target'])
y_common = df_common['target']

X_train_common, X_test, y_train_common, y_test = train_test_split(
    X_common, y_common, test_size=0.2, random_state=42, stratify=y_common
)

# 4. Manually force all the rare diseases directly into the Training Set
X_rare = df_rare.drop(columns=['target'])
y_rare = df_rare['target']

X_train = pd.concat([X_train_common, X_rare])
y_train = pd.concat([y_train_common, y_rare])

print("--- DATA PRESERVATION COMPLETE ---")
print(f"Total diseases preserved: {len(y_train.unique())}")
print(f"Training on {X_train.shape[0]} rows...")
print(f"Testing on {X_test.shape[0]} rows...")

Isolating rare diseases to protect them from the split...
--- DATA PRESERVATION COMPLETE ---
Total diseases preserved: 773
Training on 151726 rows...
Testing on 37921 rows...


In [6]:
# Define the evaluation set for live tracking
eval_set = [(X_train, y_train), (X_test, y_test)]

In [9]:
# 1. XGBoost
print("---Training XGBoost (100 Trees) ---")
t0 = time.time()

xgb_model = xgb.XGBClassifier(
    tree_method='hist',
    device='cuda',
    random_state=42,
    n_estimators=100,
    max_depth=6,
    eval_metric='mlogloss'
)

# XGBoost takes eval_set and verbose directly in the .fit() method
xgb_model.fit(
    X_train, y_train,
    eval_set=eval_set,
    verbose=1  # Prints an update every 1 tree
)
xgb_preds = xgb_model.predict(X_test)
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
acc_xgb = accuracy_score(y_test, xgb_model.predict(X_test)) * 100
prec_xgb = precision_score(y_test, xgb_preds, average='weighted', zero_division=0) * 100
rec_xgb = recall_score(y_test, xgb_preds, average='weighted', zero_division=0) * 100
f1_xgb = f1_score(y_test, xgb_preds, average='weighted', zero_division=0) * 100
results.append({
    'Model': 'XGBoost',
    'Accuracy (%)': round(acc_xgb, 2),
    'Precision (%)': round(prec_xgb, 2),
    'Recall (%)': round(rec_xgb, 2),
    'F1-Score (%)': round(f1_xgb, 2),
    'Time (Mins)': round((time.time() - t0) / 60, 2)
})

---Training XGBoost (100 Trees) ---
[0]	validation_0-mlogloss:2.89882	validation_1-mlogloss:3.04922
[1]	validation_0-mlogloss:2.24480	validation_1-mlogloss:2.37263
[2]	validation_0-mlogloss:1.84173	validation_1-mlogloss:1.99746
[3]	validation_0-mlogloss:1.49922	validation_1-mlogloss:1.65038
[4]	validation_0-mlogloss:1.22539	validation_1-mlogloss:1.37582
[5]	validation_0-mlogloss:1.06009	validation_1-mlogloss:1.21547
[6]	validation_0-mlogloss:0.92094	validation_1-mlogloss:1.08417
[7]	validation_0-mlogloss:0.81396	validation_1-mlogloss:0.97398
[8]	validation_0-mlogloss:0.74519	validation_1-mlogloss:0.90523
[9]	validation_0-mlogloss:0.69204	validation_1-mlogloss:0.85233
[10]	validation_0-mlogloss:0.64994	validation_1-mlogloss:0.81097
[11]	validation_0-mlogloss:0.61411	validation_1-mlogloss:0.77653
[12]	validation_0-mlogloss:0.58211	validation_1-mlogloss:0.74584
[13]	validation_0-mlogloss:0.55465	validation_1-mlogloss:0.72039
[14]	validation_0-mlogloss:0.53120	validation_1-mlogloss:0.69869

In [11]:
import re

# Instantly strip all illegal JSON characters from column names
X_train = X_train.rename(columns=lambda x: re.sub('[^A-Za-z0-9_]+', '_', x))
X_test = X_test.rename(columns=lambda x: re.sub('[^A-Za-z0-9_]+', '_', x))

# Re-define eval_set so it uses the newly cleaned X_test
eval_set = [(X_train, y_train), (X_test, y_test)]

print("--- Column Names Cleaned for LightGBM! ---")

--- Column Names Cleaned for LightGBM! ---


In [16]:
# 2. LightGBM
print("\n---Training LightGBM (100 Trees) ---")
t0 = time.time()

lgb_model = lgb.LGBMClassifier(
    random_state=42,
    n_estimators=100,
    max_depth=6,
    min_child_samples=1, # This allows it to learn 1-row diseases!
    n_jobs=-1
)

lgb_model.fit(
    X_train, y_train,
    eval_set=eval_set
)
lgb_preds = lgb_model.predict(X_test)
acc_lgb = accuracy_score(y_test, lgb_model.predict(X_test)) * 100
prec_lgb = precision_score(y_test, lgb_preds, average='weighted', zero_division=0) * 100
rec_lgb = recall_score(y_test, lgb_preds, average='weighted', zero_division=0) * 100
f1_lgb = f1_score(y_test, lgb_preds, average='weighted', zero_division=0) * 100
results.append({
    'Model': 'LightGBM',
    'Accuracy (%)': round(acc_lgb, 2),
    'Precision (%)': round(prec_lgb, 2),
    'Recall (%)': round(rec_lgb, 2),
    'F1-Score (%)': round(f1_lgb, 2),
    'Time (Mins)': round((time.time() - t0) / 60, 2)
})

Streaming output truncated to the last 5000 lines.
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with posit

In [14]:
# 3. GradientBoosting
print("\n---Training GradientBoosting (100 Trees) ---")
t0 = time.time()

# you turn on live updates by setting verbose=1 directly inside the model initialization.
hgb_model = HistGradientBoostingClassifier(
    random_state=42,
    max_iter=100,
    max_depth=6,
    early_stopping=False,  # This stops the internal splitting crash!
    verbose=1
)

hgb_model.fit(X_train, y_train)

# Generate predictions and calculate all metrics
hgb_preds = hgb_model.predict(X_test)
acc_hgb = accuracy_score(y_test, hgb_preds) * 100
prec_hgb = precision_score(y_test, hgb_preds, average='weighted', zero_division=0) * 100
rec_hgb = recall_score(y_test, hgb_preds, average='weighted', zero_division=0) * 100
f1_hgb = f1_score(y_test, hgb_preds, average='weighted', zero_division=0) * 100

results.append({
    'Model': 'Gradient Boosting',
    'Accuracy (%)': round(acc_hgb, 2),
    'Precision (%)': round(prec_hgb, 2),
    'Recall (%)': round(rec_hgb, 2),
    'F1-Score (%)': round(f1_hgb, 2),
    'Time (Mins)': round((time.time() - t0) / 60, 2)
})


---Training HistGradientBoosting (100 Trees) ---
Binning 0.458 GB of training data: 0.968 s
Fitting gradient boosted rounds:
Fit 77300 trees in 1251.706 s, (132770 total leaves)
Time spent computing histograms: 711.747s
Time spent finding best splits:  3.401s
Time spent applying splits:      7.126s
Time spent predicting:           16.647s


In [17]:
# FINAL TABLE
print("\n" + "="*85)
print("                           FINAL MODEL COMPARISON (100 TREES)                           ")
print("="*85)
print(pd.DataFrame(results).to_string(index=False))


                           FINAL MODEL COMPARISON (100 TREES)                           
            Model  Accuracy (%)  Precision (%)  Recall (%)  F1-Score (%)  Time (Mins)
          XGBoost         78.08          78.29       78.08         78.05         9.17
         LightGBM          0.57           0.01        0.57          0.02        21.52
Gradient Boosting          0.64           0.00        0.64          0.01        21.11
         LightGBM          0.50           1.72        0.50          0.12        20.86
